# 8 Оценка стоимости недвижимости с помощью SVR

In [37]:
#!/usr/bin/env python3
import os
import pickle

from sklearn import datasets
from sklearn.metrics import explained_variance_score, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.utils import shuffle


In [38]:
# Загрузка датасета 
data = datasets.fetch_california_housing()
# Перемешивание данных
X, y = shuffle(data.data, data.target, random_state=7)

In [39]:
# Ограничиваем размер выборки для ускорения обучения
sample_size = 20000
X_small = X[:sample_size]
y_small = y[:sample_size]

# Разбиваем данные 80/20
num_training = int(0.8 * len(X_small))

X_train = X_small[:num_training]
y_train = y_small[:num_training]

X_test = X_small[num_training:]
y_test = y_small[num_training:]

print("Обучающих объектов:", len(X_train))
print("Тестовых объектов:", len(X_test))

Обучающих объектов: 16000
Тестовых объектов: 4000


In [40]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [41]:
# Список моделей для проверки
models = [
    SVR(kernel='linear', C=1.0, epsilon=0.1),
    SVR(kernel='linear', C=1.0, epsilon=0.2),
    SVR(kernel='linear', C=10.0, epsilon=0.1),
    SVR(kernel='linear', C=10.0, epsilon=0.2),
    SVR(kernel='rbf', C=1.0, epsilon=0.1),
    SVR(kernel='rbf', C=10.0, epsilon=0.1),
    SVR(kernel='rbf', C=100.0, epsilon=0.1)
]

In [42]:
# Папка для моделей
os.makedirs("svr", exist_ok=True)

trained_models = {}

for model in models:

    # Имя файла
    model_name = f"{model.kernel}_{model.C}_{model.epsilon}"
    output_model_file = f"svr/{model_name}.pkl"

    # Обучение
    print("Обучение:", model_name)
    model.fit(X_train, y_train)

    # Сохранение
    with open(output_model_file, 'wb') as f:
        pickle.dump(model, f)

    # Сохраняем модель в словарь
    trained_models[model_name] = model

    print("Сохранено:", output_model_file)

Обучение: linear_1.0_0.1


KeyboardInterrupt: 

In [43]:
trained_models = {}

for filename in os.listdir("svr"):

    if filename.endswith(".pkl"):

        model_name = filename[:-4]
        model_path = os.path.join("svr", filename)

        with open(model_path, 'rb') as f:
            trained_models[model_name] = pickle.load(f)

print("Загруженные модели:")

for model_name in trained_models:
    print(model_name)

Загруженные модели:
linear_1.0_0.1
linear_1.0_0.2
linear_10.0_0.1
linear_10.0_0.2
rbf_1.0_0.1
rbf_10.0_0.1
rbf_100.0_0.1


In [45]:
results = []

for model_name, model in trained_models.items():

    # Предсказания
    y_test_pred = model.predict(X_test)

    # Метрики
    mse = mean_squared_error(y_test, y_test_pred)
    evs = explained_variance_score(y_test, y_test_pred)

    results.append({
        "model": model_name,
        "mse": mse,
        "evs": evs
    })

    params = model_name.split("_")
    display_name = f"{params[0]} C = {params[1]} ε = {params[2]}"
    
    print("\n####", display_name, "####")
    print("Mean squared error =", round(mse, 2))
    print("Explained variance score =", round(evs, 2))


#### linear C = 1.0 ε = 0.1 ####
Mean squared error = 2.79
Explained variance score = -1.09

#### linear C = 1.0 ε = 0.2 ####
Mean squared error = 2.57
Explained variance score = -0.93

#### linear C = 10.0 ε = 0.1 ####
Mean squared error = 2.85
Explained variance score = -1.13

#### linear C = 10.0 ε = 0.2 ####
Mean squared error = 2.6
Explained variance score = -0.95

#### rbf C = 1.0 ε = 0.1 ####
Mean squared error = 0.32
Explained variance score = 0.76

#### rbf C = 10.0 ε = 0.1 ####
Mean squared error = 0.29
Explained variance score = 0.78

#### rbf C = 100.0 ε = 0.1 ####
Mean squared error = 0.29
Explained variance score = 0.78


In [ ]:
# Несколько тестовых точек California Housing
test_points = {
    "Point 1 - Los Angeles": [
        3.5,      # MedInc
        25.0,     # HouseAge
        5.5,      # AveRooms
        1.0,      # AveBedrms
        1200.0,   # Population
        3.0,      # AveOccup
        34.05,    # Latitude
        -118.25   # Longitude
    ],

    "Point 2 - San Francisco": [
        6.0,      # MedInc
        35.0,     # HouseAge
        6.2,      # AveRooms
        1.1,      # AveBedrms
        900.0,    # Population
        2.5,      # AveOccup
        37.77,    # Latitude
        -122.42   # Longitude
    ],

    "Point 3 - San Diego": [
        4.5,      # MedInc
        20.0,     # HouseAge
        5.8,      # AveRooms
        1.0,      # AveBedrms
        1500.0,   # Population
        2.8,      # AveOccup
        32.72,    # Latitude
        -117.16   # Longitude
    ],

    "Point 4 - Sacramento": [
        3.0,      # MedInc
        30.0,     # HouseAge
        5.0,      # AveRooms
        1.0,      # AveBedrms
        1100.0,   # Population
        2.7,      # AveOccup
        38.58,    # Latitude
        -121.49   # Longitude
    ],

    "Point 5 - High income area": [
        8.0,      # MedInc
        15.0,     # HouseAge
        7.0,      # AveRooms
        1.0,      # AveBedrms
        800.0,    # Population
        2.5,      # AveOccup
        37.50,    # Latitude
        -122.30   # Longitude
    ]
}

In [ ]:
# Предсказания
for point_name, test_data in test_points.items():

    # Масштабирование точки
    test_data_scaled = scaler.transform([test_data])

    print("\n========================================")
    print(point_name)
    print("========================================")

    for model_name, model in trained_models.items():

        prediction = model.predict(test_data_scaled)[0]

        print(
            model_name,
            "->",
            round(prediction, 4)
        )


Point 1 - Los Angeles
linear_1.0_0.1 -> 1.7195
linear_1.0_0.2 -> 1.7324
linear_10.0_0.1 -> 1.7196
linear_10.0_0.2 -> 1.7323
rbf_1.0_0.1 -> 1.7607
rbf_10.0_0.1 -> 1.8622
rbf_100.0_0.1 -> 1.9924

Point 2 - San Francisco
linear_1.0_0.1 -> 3.2502
linear_1.0_0.2 -> 3.2508
linear_10.0_0.1 -> 3.25
linear_10.0_0.2 -> 3.2513
rbf_1.0_0.1 -> 3.5997
rbf_10.0_0.1 -> 3.6577
rbf_100.0_0.1 -> 3.7539

Point 3 - San Diego
linear_1.0_0.1 -> 2.2089
linear_1.0_0.2 -> 2.2166
linear_10.0_0.1 -> 2.2095
linear_10.0_0.2 -> 2.217
rbf_1.0_0.1 -> 2.0387
rbf_10.0_0.1 -> 2.0247
rbf_100.0_0.1 -> 2.0327

Point 4 - Sacramento
linear_1.0_0.1 -> 1.2126
linear_1.0_0.2 -> 1.2179
linear_10.0_0.1 -> 1.2125
linear_10.0_0.2 -> 1.2176
rbf_1.0_0.1 -> 0.9945
rbf_10.0_0.1 -> 0.9858
rbf_100.0_0.1 -> 0.97

Point 5 - High income area
linear_1.0_0.1 -> 3.8797
linear_1.0_0.2 -> 3.863
linear_10.0_0.1 -> 3.8798
linear_10.0_0.2 -> 3.8634
rbf_1.0_0.1 -> 4.2157
rbf_10.0_0.1 -> 4.3867
rbf_100.0_0.1 -> 4.491
